<div style="background:linear-gradient(135deg,#0f0c29,#302b63);padding:20px 24px;border-radius:10px;border-left:5px solid #06b6d4;font-family:Arial,sans-serif;">
  <h2 style="margin:0;color:#06b6d4;">🔍&nbsp;OpenCV Real-Time Webcam Effects</h2>
  <p style="margin:8px 0 0;color:#bbb;font-size:14px;">9 live filter modes · keyboard switching · FPS display · pure OpenCV · no external dependencies</p>
</div>

## Overview

Real-time webcam filter app using only OpenCV. Press keys **1-9** to switch effects live.

| Key | Effect |
|-----|--------|
| 1 | Cartoon (`cv2.stylization`) |
| 2 | Pencil Sketch B&W |
| 3 | Pencil Sketch Color |
| 4 | Detail Enhancement |
| 5 | Edge Preserving Filter |
| 6 | Gaussian Blur |
| 7 | Canny Edge Detection |
| 8 | Heatmap (COLORMAP_JET) |
| 9 | Ocean (COLORMAP_OCEAN) |
| q / Esc | Quit |

```bash
pip install opencv-contrib-python
```

> **Note:** Requires a webcam and display. Run the last cell, or save as `webcam_effects.py`.

## 1. Effect Engine

In [ ]:
import cv2
import numpy as np
import time

EFFECT_NAMES = {
    1: "Cartoon",
    2: "Pencil Sketch (B&W)",
    3: "Pencil Sketch (Color)",
    4: "Detail Enhancement",
    5: "Edge Preserving",
    6: "Blur",
    7: "Canny Edges",
    8: "Heatmap",
    9: "Ocean",
}

def apply_effect(frame, effect_id):
    """Apply numbered webcam filter. Returns BGR frame."""
    if effect_id == 1:
        return cv2.stylization(frame, sigma_s=150, sigma_r=0.25)
    elif effect_id == 2:
        gray, _ = cv2.pencilSketch(frame, sigma_s=60, sigma_r=0.1)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    elif effect_id == 3:
        _, color = cv2.pencilSketch(frame, sigma_s=60, sigma_r=0.1)
        return color
    elif effect_id == 4:
        return cv2.detailEnhance(frame, sigma_s=10, sigma_r=0.15)
    elif effect_id == 5:
        return cv2.edgePreservingFilter(frame, flags=1, sigma_s=50, sigma_r=0.4)
    elif effect_id == 6:
        return cv2.GaussianBlur(frame, (21, 21), 0)
    elif effect_id == 7:
        edges = cv2.Canny(frame, 100, 200)
        return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    elif effect_id == 8:
        return cv2.applyColorMap(frame, cv2.COLORMAP_JET)
    elif effect_id == 9:
        return cv2.applyColorMap(frame, cv2.COLORMAP_OCEAN)
    return frame

## 2. Helper — Overlay HUD

In [ ]:
def draw_hud(frame, effect_id, fps):
    """Draw effect name and FPS counter on the frame."""
    h, w = frame.shape[:2]
    name = EFFECT_NAMES.get(effect_id, "None")
    cv2.rectangle(frame, (0, 0), (w, 36), (0, 0, 0), -1)
    cv2.putText(frame, f"Effect {effect_id}: {name}",
                (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)
    cv2.putText(frame, f"{fps:.0f} FPS",
                (w - 90, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 1)
    cv2.putText(frame, "Press 1-9 to change | q to quit",
                (8, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1)

## ▶ Launch Webcam Stream

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Could not access webcam.")
else:
    effect_id = 1
    prev_time = time.time()

    print("Webcam stream started.")
    print("Keys: 1-9 = filter  |  q or Esc = quit")
    for k, v in EFFECT_NAMES.items():
        print(f"  {k}: {v}")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame."); break

        # FPS calculation
        now = time.time()
        fps = 1.0 / max(now - prev_time, 1e-6)
        prev_time = now

        processed = apply_effect(frame, effect_id)
        draw_hud(processed, effect_id, fps)

        cv2.imshow("OpenCV Webcam Effects", processed)

        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):
            print("Exiting..."); break
        elif key in [ord(str(i)) for i in range(1, 10)]:
            effect_id = int(chr(key))
            print(f"Effect -> {effect_id}: {EFFECT_NAMES[effect_id]}")

    cap.release()
    cv2.destroyAllWindows()
    print("Done.")